In [1]:
import pandas as pd
pd.set_option("display.max_columns", 50)
pd.set_option("display.width", 200)

ventas = pd.read_csv("../datos/crudos/ventas.csv")
print(ventas.shape)
ventas.head(20)

(60770, 10)


,folio,fecha,id_cliente,sku,cantidad,precio_unitario,descuento_pct,ruta,id_vendedor,canal
0,529565,2025-07-04,C-90104,SKU-1104,6,2020.0,10.9,R-02,V-02,Mostrador
1,559138,22/05/2023,C-01624,SKU-1124,12,17090.0,3.7,R-02,V-02,Ruta
2,537107,2025-01-30,C-02339,SKU-1047,24,9030.0,3.0,R-04,V-04,Mostrador
3,525553,31/05/2025,C-00830,SKU-1014,2,8760.0,1.0,R-07,V-07,Ruta
4,514151,08/06/2026,C-01448,SKU-1052,2,5000.0,0.1,R-02,V-02,Mostrador
5,523061,11/05/2026,C-02316,SKU-1004,6,12350.0,3.0,R-04,V-04,Ruta
6,510415,2026-05-03,C-01582,SKU-1060,24,4640.0,9.6,R-02,V-02,Televenta
7,546688,30-11-2023,C-00484,SKU-1149,1,22150.0,1.7,R-02,V-02,Televenta
8,518664,2023-05-08,C-90114,SKU-1097,24,17340.0,5.0,R-04,V-04,Ruta
9,521795,2023-12-29,C-01379,SKU-1077,24,7750.0,35.8,R-05,V-05,Ruta


In [2]:
# 1. Formatos de fecha
formatos = ["%Y-%m-%d", "%d/%m/%Y", "%d-%m-%Y", "%m/%d/%Y"]
pendientes = ventas["fecha"].copy()
for f in formatos:
    ok = pd.to_datetime(pendientes, format=f, errors="coerce").notna()
    print(f"{f}: {ok.sum():,} filas")
    pendientes = pendientes[~ok]
print(f"Sin reconocer: {len(pendientes):,}")
print(pendientes.unique()[:10])

# 2, 3. Valores imposibles
print("\nCantidades negativas:", (ventas["cantidad"] < 0).sum())
print("Valores de cantidad:", sorted(ventas["cantidad"].unique()))
print("Precio en cero:", (ventas["precio_unitario"] == 0).sum())

# 5. Integridad referencial
productos = pd.read_csv("../datos/crudos/productos.csv")
clientes = pd.read_excel("../datos/crudos/clientes.xlsx")
print("\nSKU huérfanos:", (~ventas["sku"].isin(productos["sku"])).sum())
print("Clientes huérfanos:", (~ventas["id_cliente"].isin(clientes["id_cliente"])).sum())

%Y-%m-%d: 33,206 filas
%d/%m/%Y: 16,895 filas
%d-%m-%Y: 7,974 filas
%m/%d/%Y: 2,604 filas
Sin reconocer: 91
<StringArray>
['2027-13-45']
Length: 1, dtype: str

Cantidades negativas: 403
Valores de cantidad: [np.int64(-24), np.int64(-12), np.int64(-6), np.int64(-4), np.int64(-3), np.int64(-2), np.int64(-1), np.int64(1), np.int64(2), np.int64(3), np.int64(4), np.int64(6), np.int64(12), np.int64(24)]
Precio en cero: 250

SKU huérfanos: 122
Clientes huérfanos: 181


In [3]:
print("SKUs huérfanos distintos:",
      ventas.loc[~ventas["sku"].isin(productos["sku"]), "sku"].unique())
print("Clientes huérfanos distintos:",
      ventas.loc[~ventas["id_cliente"].isin(clientes["id_cliente"]), "id_cliente"].nunique())
print("\nSKUs en catálogo que nunca se vendieron:",
      (~productos["sku"].isin(ventas["sku"])).sum())

SKUs huérfanos distintos: <StringArray>
['SKU-0000']
Length: 1, dtype: str
Clientes huérfanos distintos: 1

SKUs en catálogo que nunca se vendieron: 0


In [4]:
print("SKUs huérfanos distintos:",
      ventas.loc[~ventas["sku"].isin(productos["sku"]), "sku"].unique())
print("Clientes huérfanos distintos:",
      ventas.loc[~ventas["id_cliente"].isin(clientes["id_cliente"]), "id_cliente"].unique())
print("SKUs en catálogo que nunca se vendieron:",
      (~productos["sku"].isin(ventas["sku"])).sum())
print("Rango de fechas válidas:")
f = pd.to_datetime(ventas["fecha"], format="mixed", dayfirst=True, errors="coerce")
print(f.min(), "a", f.max())

SKUs huérfanos distintos: <StringArray>
['SKU-0000']
Length: 1, dtype: str
Clientes huérfanos distintos: <StringArray>
['C-99999']
Length: 1, dtype: str
SKUs en catálogo que nunca se vendieron: 0
Rango de fechas válidas:
2023-01-01 00:00:00 a 2026-12-06 00:00:00


In [5]:
import json
devoluciones = pd.read_csv("../datos/crudos/devoluciones.csv")
vendedores   = pd.read_excel("../datos/crudos/vendedores.xlsx")
with open("../datos/crudos/visitas_ruta.json", encoding="utf-8") as fh:
    visitas = pd.json_normalize(json.load(fh))

for nombre, df in [("CLIENTES", clientes), ("PRODUCTOS", productos),
                   ("DEVOLUCIONES", devoluciones), ("VENDEDORES", vendedores),
                   ("VISITAS", visitas)]:
    display(perfilar(df, nombre))

NameError: name 'perfilar' is not defined

In [ ]:
print("Duplicados exactos:", clientes.duplicated().sum())
print("id_cliente repetidos:", clientes["id_cliente"].duplicated().sum())
print("Ciudades distintas:", sorted(clientes["ciudad"].unique()))
print("Formatos de NIT:", clientes["nit"].head(15).tolist())
print("Correos con más de un @:", clientes["email"].astype(str).str.count("@").gt(1).sum())

In [ ]:
print(productos["categoria"].value_counts())
print("Costo >= precio:",
      (productos["costo_unitario"] >= productos["precio_lista"]).sum())

In [ ]:
## Conclusión del perfilado

De 60.770 registros de ventas, 600 son duplicados exactos y 91 tienen fecha imposible: 691 se descartan con motivo registrado. 403 corresponden a devoluciones cargadas como cantidad negativa y se reclasifican, no se eliminan. 250 con precio en cero quedan pendientes de decisión tras cruzar contra el catálogo. 122 registros apuntan a SKUs inexistentes y 181 a clientes inexistentes; se marcan y se reportan al cliente. La base útil estimada es de 60079 registros.  La base útil estimada es de 60.079 registros, de los cuales 403 se reclasifican como devoluciones y 303 quedan marcados por integridad referencial pendiente. ####

In [ ]:
descartadas = 600 + 91          # duplicados exactos + fechas imposibles
print("Base útil estimada:", len(ventas) - descartadas)

In [ ]:
ventas = ventas.drop_duplicates()

In [ ]:
ventas = ventas.drop_duplicates()   # siempre primero

def clasificar_fecha(texto):
    if "-" in texto and len(texto.split("-")[0]) == 4:
        # ¿es ISO válida o inválida? (pista: pd.to_datetime con errors="coerce")
        ...
    elif "-" in texto:
        return "dd-mm-aaaa"
    else:
        a, b, anio = texto.split("/")
        a, b = int(a), int(b)
        # aquí van tus tres casos
        ...

ventas["tipo_fecha"] = ventas["fecha"].apply(clasificar_fecha)
print(ventas["tipo_fecha"].value_counts())
print("Total:", ventas["tipo_fecha"].value_counts().sum())

In [ ]:
print(ventas["tipo_fecha"].value_counts(dropna=False))

In [ ]:
import pandas as pd
pd.set_option("display.max_columns", 50)
pd.set_option("display.width", 200)

ventas = pd.read_csv("../datos/crudos/ventas.csv")
print(ventas.shape)

In [ ]:
antes = len(ventas)
ventas = ventas.drop_duplicates()
print("Duplicados eliminados:", antes - len(ventas))
print("Filas restantes:", len(ventas))

In [ ]:
ventas_crudo = pd.read_csv("../datos/crudos/ventas.csv")
perfilar(ventas_crudo)

In [ ]:
print(ventas_crudo.groupby("id_vendedor")["ruta"].nunique())

In [ ]:
print(ventas_crudo.groupby("ruta")["id_vendedor"].nunique())

In [ ]:
import json

productos    = pd.read_csv("../datos/crudos/productos.csv")
devoluciones = pd.read_csv("../datos/crudos/devoluciones.csv")
clientes     = pd.read_excel("../datos/crudos/clientes.xlsx")
vendedores   = pd.read_excel("../datos/crudos/vendedores.xlsx")

with open("../datos/crudos/visitas_ruta.json", encoding="utf-8") as f:
    visitas = pd.json_normalize(json.load(f))

for nombre, df in [("productos", productos), ("devoluciones", devoluciones),
                   ("clientes", clientes), ("vendedores", vendedores),
                   ("visitas", visitas)]:
    print(f"\n===== {nombre}: {df.shape[0]:,} filas x {df.shape[1]} columnas =====")
    display(perfilar(df))

In [ ]:
for tipo in ["barra: día/mes seguro", "barra: mes/día seguro", "barra: ambigua"]:
    ejemplos = ventas.loc[ventas["tipo_fecha"] == tipo, "fecha"].head(5).tolist()
    print(tipo, "→", ejemplos)

In [ ]:
conteo = ventas["tipo_fecha"].value_counts()

ambiguas = conteo["barra: ambigua"]
base_util = len(ventas) - conteo["inválida"]
print(f"Ambiguas: {ambiguas:,} = {ambiguas / base_util:.1%} de la base útil")

dm = conteo["barra: día/mes seguro"]
md = conteo["barra: mes/día seguro"]
print(f"Entre las seguras: día/mes {dm / (dm + md):.0%} | mes/día {md / (dm + md):.0%}")

In [ ]:
import os
print("Estoy en:", os.getcwd())
print("¿Existe la carpeta scripts?", os.path.exists("../scripts"))
if os.path.exists("../scripts"):
    print("Contenido de scripts:", os.listdir("../scripts"))

In [ ]:
import os
print(os.listdir("../scripts"))

In [ ]:
print(os.listdir("../scripts"))

In [8]:
import glob, os, shutil

candidatos = glob.glob(os.path.expanduser("~/Downloads/herramientas_perfilado*"))
print("Encontrados:", candidatos)

if candidatos:
    shutil.copy(candidatos[0], "../scripts/herramientas_perfilado.py")
    print("Copiado. Contenido de scripts:", os.listdir("../scripts"))

Encontrados: []


In [9]:
import os, sys
print(os.listdir("../scripts"))

sys.path.append("../scripts")
from herramientas_perfilado import perfilar, patrones, huerfanos, clasificar_fecha
print("Importado correctamente")

['.ipynb_checkpoints', '02_limpieza.py', 'herramientas_perfilado.py', '__pycache__']
Importado correctamente


In [7]:
import pandas as pd
import os, sys
sys.path.append("../scripts")
from herramientas_perfilado import perfilar, patrones, huerfanos, clasificar_fecha